In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display


def generate_positional_encoding(max_seq_len, d_model):
    """Generates the Positional Encoding matrix."""
    pe = np.zeros((max_seq_len, d_model))
    position = np.arange(0, max_seq_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(1000000.0) / d_model))

    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)

    return pe


def display_graphs(max_seq_len, d_model):
    """Function called by the widget to update the display."""
    # Ensure that d_model is always an even number
    if d_model % 2 != 0:
        d_model += 1

    pe_matrix = generate_positional_encoding(max_seq_len, d_model)

    plt.figure(figsize=(16, 6))

    # 1. Heatmap (Global matrix)
    plt.subplot(1, 2, 1)
    plt.pcolormesh(pe_matrix, cmap="RdBu", vmin=-1, vmax=1)
    plt.title(
        f"Positional Encoding Matrix\n(d_model={d_model}, seq_len={max_seq_len})"
    )
    plt.xlabel("Dimensions (0 to d_model)")
    plt.ylabel("Token position")
    plt.colorbar(label="Value")
    plt.gca().invert_yaxis()

    # 2. Evolution curves for the first dimensions
    plt.subplot(1, 2, 2)

    # Plot the first 4 dimensions to see the sin/cos alternation
    # and the frequency difference. Ensure the size allows it.
    if d_model >= 4:
        plt.plot(
            pe_matrix[:, 0],
            label="Dim 0 (Sine, very fast)",
            color="#1f77b4",
            linewidth=2,
        )
        plt.plot(
            pe_matrix[:, 1],
            label="Dim 1 (Cosine, very fast)",
            color="#ff7f0e",
            linewidth=2,
            linestyle="--",
        )
        plt.plot(
            pe_matrix[:, 2],
            label="Dim 2 (Sine, slightly slower)",
            color="#2ca02c",
            linewidth=2,
        )
        plt.plot(
            pe_matrix[:, 3],
            label="Dim 3 (Cosine, slightly slower)",
            color="#d62728",
            linewidth=2,
            linestyle="--",
        )

    plt.title("Oscillations of the first dimensions")
    plt.xlabel("Token position in the sequence")
    plt.ylabel("Value (-1 to 1)")
    plt.legend(loc="upper right")
    plt.grid(True, alpha=0.4)

    plt.tight_layout()
    plt.show()


# --- WIDGET CREATION ---
# Create sliders to manipulate values live
widget_seq_len = widgets.IntSlider(
    value=50,
    min=10,
    max=200,
    step=10,
    description="Tokens number:",
    continuous_update=False,  # Updates the graph only when the click is released
)

widget_d_model = widgets.IntSlider(
    value=64,
    min=16,
    max=512,
    step=16,
    description="Dimensions:",
    continuous_update=False,
)

# Link the display function to the widgets
ui = widgets.interact(
    display_graphs, max_seq_len=widget_seq_len, d_model=widget_d_model
)

interactive(children=(IntSlider(value=50, continuous_update=False, description='Tokens number:', max=200, min=…

### Animation : faire varier `seq_len`, `d_model` **et** `theta_base`

L'encodage positionnel sinusoïdal a **trois** leviers, et l'animation ci-dessous les fait bouger l'un après l'autre :

| Paramètre | Rôle | Effet visuel |
|-----------|------|--------------|
| `seq_len` | nombre de tokens (positions) | la matrice s'allonge verticalement |
| `d_model` | profondeur de l'embedding | plus de colonnes, donc plus de fréquences |
| `theta_base` | la fameuse constante `10000` de la formule `1 / theta_base^(2i/d)` | pilote l'**étendue des longueurs d'onde** |

**Pourquoi faire varier `theta_base` ?** C'est lui qui fixe l'écart entre la dimension qui tourne le plus vite et celle qui tourne le plus lentement :

- **`theta_base` petit** → toutes les dimensions oscillent vite → on distingue bien des positions **proches**, mais on « sature » vite sur de longues séquences.
- **`theta_base` grand** → apparition de dimensions très **lentes** (bandes quasi-constantes à droite de la matrice) → on peut encoder des positions **très éloignées** sans ambiguïté.

C'est exactement ce levier que des modèles comme **LLaMA** augmentent (par ex. `500000`) pour étendre la fenêtre de contexte. La 3ᵉ phase de l'animation balaie `theta_base` de `100` à `1 000 000` (échelle log) pour rendre cet effet visible.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

# 1. Base function
# theta_base is the constant at the heart of the formula (the famous "10000").
# It sets the RANGE of wavelengths used to encode positions:
#   - small theta_base  -> all dimensions oscillate fast  -> good for short contexts
#   - large theta_base  -> slow dimensions appear         -> needed for long contexts
# Models like LLaMA increase it (e.g. 500000) precisely to handle long sequences.
def generate_positional_encoding(max_seq_len, d_model, theta_base=10000.0):
    pe = np.zeros((max_seq_len, d_model))
    position = np.arange(0, max_seq_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(theta_base) / d_model))

    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe

# 2. Figure preparation
fig, ax = plt.subplots(figsize=(10, 6.5))
# Reserve room at the top for the 3-line title (set once: ax.clear() keeps it)
fig.subplots_adjust(top=0.80, bottom=0.12)

# 3. Animation scenario (the "frames")
# Each frame carries the moving parameter so the caption can explain it.
# (phase, seq_len, d_model, theta_base)
# Phase 1: seq_len varies, d_model=64, theta=10000
frames_phase1 = [("seq_len", seq, 64, 10000.0) for seq in range(10, 102, 2)]
# Phase 2: d_model varies, seq_len=100, theta=10000
frames_phase2 = [("d_model", 100, dim, 10000.0) for dim in range(16, 130, 4)]
# Phase 3: theta_base varies (log scale), seq_len=100, d_model=64
thetas = np.geomspace(100, 1_000_000, 40)
frames_phase3 = [("theta", 100, 64, float(t)) for t in thetas]
# Combine the three phases
all_frames = frames_phase1 + frames_phase2 + frames_phase3

# Short explanation shown for each phase
explications = {
    "seq_len": "Phase 1 - on fait varier le nombre de tokens (seq_len) : la matrice s'allonge.",
    "d_model": "Phase 2 - on fait varier la profondeur (d_model) : plus de colonnes/frequences.",
    "theta":   "Phase 3 - on fait varier la base theta : elle pilote l'etendue des longueurs d'onde.",
}

# 4. Update function for each GIF frame
def update(frame_data):
    ax.clear()  # Clear the previous image
    phase, seq_len, d_model, theta_base = frame_data

    pe_matrix = generate_positional_encoding(seq_len, d_model, theta_base)

    # Draw the matrix
    mesh = ax.pcolormesh(pe_matrix, cmap='RdBu', vmin=-1, vmax=1)

    titre = (
        f"Animated Positional Encoding\n"
        f"seq_len = {seq_len}  |  d_model = {d_model}  |  theta_base = {theta_base:,.0f}\n"
        f"{explications[phase]}"
    )
    ax.set_title(titre, fontsize=11)
    ax.set_xlabel('Dimension depth')
    ax.set_ylabel('Token position')
    ax.invert_yaxis()

    return [mesh]

# 5. Animation creation
print("Generating animation, please wait...")
ani = FuncAnimation(fig, update, frames=all_frames, interval=100, blit=False)

# 6. Save as GIF file
filename = "evolution_positional_encoding.gif"
writer = PillowWriter(fps=10) # 10 frames per second
ani.save(filename, writer=writer)

print(f"GIF successfully generated! Saved as: '{filename}'")

# Optional: Display the animation directly in the notebook (HTML)
plt.close() # Prevents displaying a duplicate static image
HTML(ani.to_jshtml())

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib.patches import Arc

def afficher_rotation_rope(m, theta_deg):
    """
    Simule et affiche l'effet de RoPE sur une paire de dimensions.
    
    Arguments:
    - m : La position du token dans la phrase.
    - theta_deg : L'angle de base (Theta) en degrés.
    """
    # 1. On imagine un vecteur sémantique pur pour notre mot (ex: "chat")
    # Pour simplifier la vue 2D, on le place à l'horizontale sur (1, 0)
    v_init = np.array([1.0, 0.0])
    
    # Conversion de Theta en radians pour les mathématiques
    theta = np.radians(theta_deg)
    
    # 2. Le cœur de RoPE : L'angle total est la position multipliée par Theta
    angle_total = m * theta
    
    # 3. Création de la matrice de rotation
    R = np.array([
        [np.cos(angle_total), -np.sin(angle_total)],
        [np.sin(angle_total),  np.cos(angle_total)]
    ])
    
    # 4. On applique la rotation au vecteur (Produit matriciel)
    v_rot = np.dot(R, v_init)
    
    # --- VISUALISATION ---
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Dessin d'un cercle unitaire en fond pour repère
    cercle = plt.Circle((0, 0), 1, color='lightgray', fill=False, linestyle='--')
    ax.add_patch(cercle)
    
    # Axes X et Y
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    
    # Dessin du vecteur initial (Sémantique sans position)
    ax.quiver(0, 0, v_init[0], v_init[1], angles='xy', scale_units='xy', scale=1, 
              color='blue', width=0.01, label='Vecteur Initial (Sémantique)')
    
    # Dessin du vecteur après RoPE (Sémantique + Position)
    ax.quiver(0, 0, v_rot[0], v_rot[1], angles='xy', scale_units='xy', scale=1, 
              color='red', width=0.01, label=f'Vecteur RoPE (Position {m})')
    
    # Ajout d'un arc pour montrer l'angle
    if m > 0:
        # L'arc de matplotlib prend le diamètre (2) et les angles en degrés
        arc = Arc((0,0), 0.5, 0.5, angle=0, theta1=0, theta2=np.degrees(angle_total), color='green', linewidth=2)
        ax.add_patch(arc)
        ax.text(0.3 * np.cos(angle_total/2), 0.3 * np.sin(angle_total/2), 
                f'{m * theta_deg}°', color='green', fontsize=12, ha='center', va='center', weight='bold')

    # Formatage du graphique
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_aspect('equal') # Pour que le cercle soit bien rond
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1))
    ax.set_title(f"RoPE : Rotation d'une paire de dimensions\nAngle total = m × Theta = {m} × {theta_deg}° = {m * theta_deg}°", fontsize=14)
    
    plt.tight_layout()
    plt.show()

# --- CRÉATION DES WIDGETS ---
widget_m = widgets.IntSlider(
    value=1, min=0, max=24, step=1, 
    description='Position (m):',
    continuous_update=False
)

widget_theta = widgets.IntSlider(
    value=15, min=5, max=90, step=5, 
    description='Theta (°):',
    continuous_update=False
)

# Affichage interactif
ui = widgets.interact(afficher_rotation_rope, m=widget_m, theta_deg=widget_theta)

interactive(children=(IntSlider(value=1, continuous_update=False, description='Position (m):', max=24), IntSli…

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

# --- PARAMÈTRES RÉALISTES ---
m_max = 20 # On va lire une phrase de 20 mots (positions 0 à 20)

# On va simuler 3 paires de dimensions de notre vecteur d'embedding.
# Dans la réalité, ces angles (theta) sont calculés avec la formule du 10000.
# Pour que l'animation soit lisible, on choisit 3 angles bien distincts :
vitesses = {
    "Paire 1 (Dim 0-1)": 45,  # Tourne très vite (45° à chaque mot)
    "Paire 2 (Dim 4-5)": 15,  # Tourne moyennement (15° à chaque mot)
    "Paire 3 (Dim 62-63)": 3  # Tourne très lentement (3° à chaque mot)
}
couleurs = ['#d62728', '#1f77b4', '#2ca02c'] # Rouge, Bleu, Vert

# --- PRÉPARATION DU GRAPHIQUE ---
fig, ax = plt.subplots(figsize=(8, 8))

def update(m):
    ax.clear()
    
    # Dessin du cadran
    cercle = plt.Circle((0, 0), 1, color='lightgray', fill=False, linestyle='--')
    ax.add_patch(cercle)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    
    # On calcule et on dessine la rotation pour CHAQUE paire
    for i, (nom_paire, angle_par_mot) in enumerate(vitesses.items()):
        # Le vrai calcul mathématique : Position (m) * Vitesse de la paire (theta)
        angle_total_deg = m * angle_par_mot
        angle_rad = np.radians(angle_total_deg)
        
        # Calcul des coordonnées (x, y) après rotation
        v_x = np.cos(angle_rad)
        v_y = np.sin(angle_rad)
        
        # Dessin du vecteur
        ax.quiver(0, 0, v_x, v_y, angles='xy', scale_units='xy', scale=1, 
                  color=couleurs[i], width=0.01, label=f"{nom_paire} ({angle_total_deg}°)")
    
    # Formatage
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_aspect('equal')
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1))
    
    # Titre montrant qu'on avance par nombres entiers stricts
    ax.set_title(f"RoPE : Traitement par paires de dimensions\nLecture du Token à la Position : {m}", 
                 fontsize=14, fontweight='bold')

# --- CRÉATION DE L'ANIMATION ---
print("Génération de l'animation exacte (positions discrètes)...")

# 'frames' utilise range(m_max + 1) -> Ce qui génère exactement [0, 1, 2, 3... 20]
ani = FuncAnimation(fig, update, frames=range(m_max + 1), interval=600) # interval=600ms par bond

nom_fichier = "rope_paires_discretes.gif"
writer = PillowWriter(fps=1.5) # Animation volontairement saccadée pour bien voir les bonds
ani.save(nom_fichier, writer=writer)

print(f"GIF généré ! Fichier : '{nom_fichier}'")

plt.close()
HTML(ani.to_jshtml())

Génération de l'animation exacte (positions discrètes)...
GIF généré ! Fichier : 'rope_paires_discretes.gif'


In [5]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def afficher_rope_paires(m):
    """
    Affiche la rotation discrète de 3 paires de dimensions selon la position m.
    """
    # Nos 3 paires fictives avec leurs vitesses de rotation (Theta)
    paires = [
        {"nom": "Paire Rapide (ex: Dim 0-1)", "theta": 45, "couleur": "#d62728"}, # Rouge
        {"nom": "Paire Moyenne (ex: Dim 4-5)", "theta": 15, "couleur": "#1f77b4"}, # Bleu
        {"nom": "Paire Lente (ex: Dim 62-63)", "theta": 3,  "couleur": "#2ca02c"}  # Vert
    ]
    
    fig, ax = plt.subplots(figsize=(8, 7))
    
    # Dessin du cadran de base (Cercle et axes X/Y)
    cercle = plt.Circle((0, 0), 1, color='lightgray', fill=False, linestyle='--')
    ax.add_patch(cercle)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    
    textes_info = [] # Pour stocker les infos à afficher sous le graphe
    
    # Boucle pour calculer et dessiner le vecteur de chaque paire
    for p in paires:
        # VRAI CALCUL ROPE : Angle = position * angle_de_base
        angle_deg = m * p["theta"]
        angle_rad = np.radians(angle_deg)
        
        # Coordonnées du vecteur
        v_x = np.cos(angle_rad)
        v_y = np.sin(angle_rad)
        
        # Tracé de la flèche
        ax.quiver(0, 0, v_x, v_y, angles='xy', scale_units='xy', scale=1, 
                  color=p["couleur"], width=0.008, label=f'{p["nom"]} : {angle_deg}°')
        
        # Préparation du texte pour le récapitulatif
        textes_info.append(f"{p['nom']:<27} | Vitesse : {p['theta']:>2}°/mot | Angle Total : {angle_deg}°")
        
    # Formatage visuel du graphique
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_aspect('equal')
    ax.grid(True, linestyle=':', alpha=0.5)
    
    # Placement de la légende en dessous pour éviter qu'elle soit coupée sur les côtés
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), fancybox=True, shadow=True, ncol=1)
    ax.set_title(f"RoPE : Position du Token (m) = {m}", fontsize=15, fontweight='bold', pad=15)
    
    plt.tight_layout()
    plt.show()
    
    # Affichage en texte brut sécurisé sous le widget
    print("=" * 65)
    for texte in textes_info:
        print(texte)
    print("=" * 65)

# --- CRÉATION DU WIDGET INTERACTIF ---
# Un slider qui force l'utilisateur à rester sur des entiers stricts (0, 1, 2, 3...)
slider_m = widgets.IntSlider(
    value=0, 
    min=0, 
    max=30, 
    step=1, 
    description='Position m :',
    layout=widgets.Layout(width='600px'),
    continuous_update=False # Met à jour quand on relâche la souris pour éviter le lag
)

# On connecte le slider à la fonction
ui = widgets.interact(afficher_rope_paires, m=slider_m)

interactive(children=(IntSlider(value=0, continuous_update=False, description='Position m :', layout=Layout(wi…

In [ ]:
import numpy as np
# import matplotlib.subplots
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML

# --- REALISTIC PARAMETERS ---
m_max = 20 # We will read a sequence of 20 tokens (positions 0 to 20)

# We will simulate 3 pairs of dimensions from our embedding vector.
# In reality, these base angles (theta) are calculated using the 10000 formula.
# For a readable animation, we choose 3 distinct angles:
speeds = {
    "Fast Pair (Dim 0-1)": 45,   # Turns very fast (45° per position)
    "Medium Pair (Dim 4-5)": 15, # Turns moderately (15° per position)
    "Slow Pair (Dim 62-63)": 3   # Turns very slowly (3° per position)
}
colors = ['#d62728', '#1f77b4', '#2ca02c'] # Red, Blue, Green

# --- GRAPH SETUP ---
fig, ax = plt.subplots(figsize=(9, 8))

# Reserve space on the right so the legend (placed outside the axes) is not
# cropped when the figure is rasterized by PillowWriter. ax.clear() in update()
# does not reset this margin, so calling it once here is enough.
fig.subplots_adjust(left=0.08, right=0.68, top=0.88, bottom=0.08)

def update(m):
    ax.clear()
    
    # Draw the base dial (Unit circle)
    circle = plt.Circle((0, 0), 1, color='lightgray', fill=False, linestyle='--')
    ax.add_patch(circle)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    
    # Calculate and draw the rotation for EACH pair
    for i, (pair_name, angle_per_word) in enumerate(speeds.items()):
        # The core RoPE calculation: Position (m) * Pair Speed (theta)
        total_angle_deg = m * angle_per_word
        angle_rad = np.radians(total_angle_deg)
        
        # Calculate (x, y) coordinates after rotation
        v_x = np.cos(angle_rad)
        v_y = np.sin(angle_rad)
        
        # Draw the vector
        ax.quiver(0, 0, v_x, v_y, angles='xy', scale_units='xy', scale=1, 
                  color=colors[i], width=0.01, label=f"{pair_name}: {total_angle_deg}°")
    
    # Formatting the plot
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_aspect('equal')
    ax.grid(True, linestyle=':', alpha=0.6)
    
    # Legend placed outside (right) to avoid overlapping the vectors
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1))
    
    # Title showing strict integer progression
    ax.set_title(f"RoPE: Discrete Dimension Pairs\nProcessing Token at Position (m): {m}", 
                 fontsize=14, fontweight='bold', pad=15)

# --- CREATE AND SAVE THE ANIMATION ---
print("Generating the discrete animation...")

# 'frames' uses range(m_max + 1) -> Generates exactly [0, 1, 2, 3... 20]
ani = FuncAnimation(fig, update, frames=range(m_max + 1), interval=600)

# Save as GIF
filename = "rope_discrete_pairs_english.gif"
writer = PillowWriter(fps=1.5) # Low FPS so the discrete jumps are clearly visible
ani.save(filename, writer=writer)

print(f"Success! The GIF has been saved as: '{filename}'")

# Display inline in the notebook
plt.close() # Prevent a duplicate static plot from rendering
HTML(ani.to_jshtml())
